### ENV Setup

In [3]:
MODEL_ID_LIST = ['tavakolih/all-MiniLM-L6-v2-pubmed-full', 'Alibaba-NLP/gte-Qwen2-1.5B-instruct', 'google/medgemma-27b-text-it']

In [1]:
from owlready2 import *

In [ ]:
output = ''

In [2]:
vo = get_ontology("vo_source/VO_merged2022_10_05.owl").load()
vv = IRIS['http://purl.obolibrary.org/obo/VO_0004815']
# rows = []
# for cls in vo.classes():
#     name = cls.name
#     if name.startswith("VO_"):
#         label_text = cls.label.en.first() if isinstance(cls.label.first(), locstr) else cls.label.first()
#         rows.append({"vo_id": name, "label": label_text})
# vo_df = pd.DataFrame(rows)

In [10]:
type(vv.VO_0003198.first())

int

In [6]:
hasattr(vv, 'VO_0003198')

True

In [5]:
VO_0003198[vv]

NameError: name 'VO_0003198' is not defined

In [5]:
from typing import List, Sequence, Tuple, Optional
import numpy as np
import faiss
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
from sentence_transformers import SentenceTransformer
import requests
import pandas as pd
import pickle

In [6]:
def load_model(model_id):
    print(f"Loading model {model_id}...")
    model = SentenceTransformer(model_id, device='cuda', model_kwargs={'device_map': "auto"})
    print(f"Model {model_id} loaded to CUDA {model.device} successfully.")
    return model

### Dense Index

In [7]:
# encode texts with the model with parameter for batch size
def encode_texts(model, texts, batch_size=32, normalize=True):
    embeddings = model.encode(
        texts, 
        batch_size=batch_size,
        show_progress_bar=True, 
        convert_to_numpy=True,
        normalize_embeddings=normalize,
    )
    return embeddings.astype("float32") # FAISS needs float32

In [8]:
#build faiss index
def build_faiss_index(embeddings:np.ndarray,ids: Optional[Sequence[int]] = None,use_gpu: bool = True) -> faiss.Index:
    import faiss
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product of normalized vectors is  ~ cosine similarity
    if use_gpu and faiss.get_num_gpus() > 0:
        res   = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
    if ids is not None:
        id_map = faiss.IndexIDMap(index)
        id_map.add_with_ids(embeddings, np.asarray(ids, dtype="int64"))
        print(f"Index built with {len(ids)} IDs.")
        return id_map
    else:
        index.add(embeddings)
        print(f"Index built with {embeddings.shape[0]} vectors.")
        return index

In [ ]:
# # SMOKE TEST 
# rx_labels = [
#         "Influenza A (H1N1) vaccine, inactivated",
#         "Measles, Mumps and Rubella vaccine, live",
#         "COVID-19 mRNA-LNP vaccine, bivalent",
#     ]

In [ ]:
# D, I = faiss_index.search(vecs[0:1], k=3)   # cosine scores + indices
# print("Nearest neighbours:", I, "scores:", D)

Nearest neighbours: [[1 0 3]] scores: [[1.         1.         0.43668175]]


### BM25 Index

### 1. ElastiSearch Env Load

In [9]:
import time, requests, subprocess
from pathlib import Path
ES_HOME = Path("/data/wmanuel3/VaxMapperRepo/esdata/elasticsearch-9.0.0")
ES_PORT  = 9200
# PID_FILE  = ES_HOME / "run/es.pid"
# LOG_FILE  = ES_HOME / "logs/console.log"

In [10]:
import subprocess
import os
import signal
import time
from pathlib import Path

def run_elasticsearch():
    # Path to elasticsearch executable
    es_executable = ES_HOME / "bin/elasticsearch"
    
    # Environment variables for Elasticsearch
    env = os.environ.copy()
    env.update({
        "ES_JAVA_OPTS": "-Xms2g -Xmx2g",
        "discovery.type": "single-node",
        "xpack.security.enabled": "false"
    })
    
    print(f"Starting Elasticsearch from {es_executable}...")
    
    # Run Elasticsearch in background
    process = subprocess.Popen(
        [str(es_executable)],
        env=env,
        cwd=ES_HOME,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    
    # Wait for Elasticsearch to start
    print(f"Elasticsearch started with PID: {process.pid}")
    print("Waiting for service to become available...")
    
    # Try connecting to verify it's up
    for _ in range(60):
        try:
            response = requests.get(f"http://localhost:{ES_PORT}", timeout=2)
            if response.status_code == 200:
                print(f"Elasticsearch is running: {response.json().get('version', {}).get('number')}")
                return process
        except (requests.ConnectionError, requests.Timeout):
            print(".", end="", flush=True)
            time.sleep(2)
    
    print("Elasticsearch didn't start properly within the timeout period")
    return process

# Run it
es_process = run_elasticsearch()

Starting Elasticsearch from /data/wmanuel3/VaxMapperRepo/esdata/elasticsearch-9.0.0/bin/elasticsearch...
Elasticsearch started with PID: 1175589
Waiting for service to become available...
Elasticsearch is running: 9.0.0


In [ ]:
es_process.terminate()
time.sleep(5)
if es_process.poll() is None:  # Still running
    print("Sending SIGKILL...")
    es_process.kill()

In [11]:
from elasticsearch import Elasticsearch
es = Elasticsearch("http://localhost:9200")
es.info()

ObjectApiResponse({'name': 'msdplaplp001.uth.tmc.edu', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'm4zPSJyjTKCUH2PiE0ADfw', 'version': {'number': '9.0.0', 'build_flavor': 'default', 'build_type': 'tar', 'build_hash': '112859b85d50de2a7e63f73c8fc70b99eea24291', 'build_date': '2025-04-08T15:13:46.049795831Z', 'build_snapshot': False, 'lucene_version': '10.1.0', 'minimum_wire_compatibility_version': '8.18.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

Refs: 
* Analyzer [https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/analyzer]
* Syns graph [https://www.elastic.co/docs/reference/text-analysis/analysis-synonym-graph-tokenfilter]
* Similarity [https://www.elastic.co/docs/reference/elasticsearch/index-settings/similarity]
* Mapping []

Todo: Apply synonyms during indexing or search ?


In [13]:
INDEX = "rxnorm_bm25"
settings = {
    "settings": {
        "analysis": {  # Analysis settings for synonyms and tokenization
            "filter": {
                "vaccine_syns": {
                    "type": "synonym_graph",
                    "synonyms_path": "vaccine_syns.txt"
                }
            },
            "analyzer": {
                "vaccine_analyzer": {
                    "tokenizer": "standard",
                    "filter": ["lowercase", "asciifolding", "vaccine_syns"]
                }
            }
        },
        "index": {
            "similarity": {
                "bm25_short": {         # tune for short biomedical strings
                    "type": "BM25",
                    "k1": 0.9,            # slightly below default 1.2
                    "b": 0.4              # lighter length normalisation than 0.75
                }
            }
        }
    },
    "mappings": { # define the document level structure
        "properties": {
            "rxcui": {"type": "keyword"},
            "tty": {"type": "keyword"},
            "text": {
                "type": "text",
                "analyzer": "vaccine_analyzer",
                "similarity": "bm25_short"
            }
        }
    }
}

es.indices.create(index=INDEX, body=settings, ignore=400)

/tmp/ipykernel_1140301/3070531291.py:41: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es.indices.create(index=INDEX, body=settings, ignore=400)


ObjectApiResponse({'error': {'root_cause': [{'type': 'resource_already_exists_exception', 'reason': 'index [rxnorm_bm25/oyEaPa75QWi17PCIVJ4rgw] already exists', 'index_uuid': 'oyEaPa75QWi17PCIVJ4rgw', 'index': 'rxnorm_bm25'}], 'type': 'resource_already_exists_exception', 'reason': 'index [rxnorm_bm25/oyEaPa75QWi17PCIVJ4rgw] already exists', 'index_uuid': 'oyEaPa75QWi17PCIVJ4rgw', 'index': 'rxnorm_bm25'}, 'status': 400})

In [12]:
VO_INDEX = "vo_bm25"

vo_settings = {
  "settings": {
    "analysis": {
      "filter": {
        "vaccine_syns": {                
          "type": "synonym_graph",
          "synonyms_path": "vaccine_syns.txt"
        }
      },
      "analyzer": {
        "vaccine_analyzer": {
          "tokenizer": "standard",
          "filter": ["lowercase", "asciifolding", "vaccine_syns"]
        }
      }
    },
    "similarity": {
      "bm25_short": {
        "type": "BM25",
        "k1": 0.9,
        "b":  0.4
      }
    }
  },
  "mappings": {
    "properties": {
      "vo_id":  { "type": "keyword" },        
      "iri":    { "type": "keyword" },        
      "text": {
        "type": "text",
        "analyzer":   "vaccine_analyzer",
        "similarity": "bm25_short"
      }
    }
  }
}

es.indices.create(index=VO_INDEX, body=vo_settings, ignore=400)


/tmp/ipykernel_1175485/1363736000.py:40: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es.indices.create(index=VO_INDEX, body=vo_settings, ignore=400)


ObjectApiResponse({'error': {'root_cause': [{'type': 'resource_already_exists_exception', 'reason': 'index [vo_bm25/Zl0Cjt-6SdOqLKR83kU_rw] already exists', 'index_uuid': 'Zl0Cjt-6SdOqLKR83kU_rw', 'index': 'vo_bm25'}], 'type': 'resource_already_exists_exception', 'reason': 'index [vo_bm25/Zl0Cjt-6SdOqLKR83kU_rw] already exists', 'index_uuid': 'Zl0Cjt-6SdOqLKR83kU_rw', 'index': 'vo_bm25'}, 'status': 400})

### 2. Loading RxNorm and indexing: 

In [14]:
# Helper functions from rxnorm_getter 
# Todo: Move them out of the identify_missing_rxnorm_terms function: 

def get_rxnorm_data(rxcui):
        result = {}
        result['rxcui'] = rxcui
        status_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/historystatus.json?"
        try:
            response = requests.get(status_url)
            if response.status_code == 200:
                data = response.json()
                result['name'] = data.get("rxcuiStatusHistory", {}).get("attributes", {}).get("name")
                result['status'] = data.get("rxcuiStatusHistory", {}).get("metaData", {}).get("status")
                result['tty'] = data.get("rxcuiStatusHistory", {}).get("attributes", {}).get("tty")
        except requests.RequestException as e:
            print(f"Error fetching RxNorm status for {rxcui}: {e}")
        return result

def get_class_descendants(class_id, class_type):
    api_url = f"https://rxnav.nlm.nih.gov/REST/rxclass/classTree.json?classId={class_id}&classType={class_type}"
    descendants = []
    
    try:
        response = requests.get(api_url)
        response.raise_for_status()      
        data = response.json()
        descendants = _collect_leaf_classids(data["rxclassTree"])
        
    except Exception as e:
        print(f"API error with {class_id}: {e}")
        
    return descendants

def _collect_leaf_classids(tree):
    leaves = []
    for node in tree:
        cid = node["rxclassMinConceptItem"]["classId"]
        cname = node["rxclassMinConceptItem"]["className"]
        children = node.get("rxclassTree")
        if children:                      
            leaves.extend(_collect_leaf_classids(children))
        else:                             
            leaves.append({'ID': cid, 'Name': cname})
    return leaves

def get_rx_class_members(params):
    api_url = f"https://rxnav.nlm.nih.gov/REST/rxclass/classMembers.json"
    class_id = params.get("classId")
    source = params.get("relaSource")
    result = {"class_id": class_id, "source": source, "related_ids": None}
    
    try:
        response = requests.get(api_url, params=params)
        response.raise_for_status()      
        data = response.json()
        if "drugMemberGroup" in data and "drugMember" in data["drugMemberGroup"]:
            related_ids = [item["minConcept"]["rxcui"] for item in data["drugMemberGroup"]["drugMember"]]
            result["related_ids"] = related_ids
    except Exception as e:
        print(f"API error with {class_id}: {e}")

    return result

def get_related_rxcui(rxcui):
    api_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/allrelated.json"
    result = {"rxcui": rxcui, "related_ids": None}

    try:
        response = requests.get(api_url)
        response.raise_for_status()      
        data = response.json()
        concept_groups = data.get("allRelatedGroup", {}).get("conceptGroup", [])
        related_concepts = []
        for group in concept_groups:
            if "conceptProperties" in group:
                related_concepts.extend([concept.get("rxcui") for concept in group["conceptProperties"]])

        result["related_ids"] = related_concepts

    except Exception as e:
        print(f"API error with {rxcui}: {e}")

    return result

In [15]:
# Collect RxNorm concepts from different sources
rx_concepts = set()

# ATC concepts
print("Collecting from ATC...")
atc_descendants = get_class_descendants("J07", "ATC1-4")
for atc in atc_descendants:
    params = {"classId": atc['ID'], "relaSource": "ATCPROD"}
    res = get_rx_class_members(params)
    if res['related_ids']:
        rx_concepts.update(res['related_ids'])

# VA concepts
print("Collecting from VA...")
VA_Classes = ["IM100", "IM105", "IM109"]
rela = ["has_vaclass", "has_vaclass_extended"]
for va in VA_Classes:
    for r in rela:
        params = {"classId": va, "relaSource": "VA", "rela": r}
        res = get_rx_class_members(params)
        if res['related_ids']:
            rx_concepts.update(res['related_ids'])

# CVX concepts
print("Collecting from CVX...")
cvx_descendants = get_class_descendants("0", "CVX")
for cvx in cvx_descendants:
    params = {"classId": cvx['ID'], "relaSource": "CDC", "rela": "isa_CVX"}
    res = get_rx_class_members(params)
    if res['related_ids']:
        rx_concepts.update(res['related_ids'])

# DailyMed concepts
print("Collecting from DailyMed...")
dm_classes = ["N0000193912", "D014612"]
rela = ["has_epc", "has_chemical_structure"]
for dm in dm_classes:
    for r in rela:
        params = {"classId": dm, "relaSource": "DAILYMED", "rela": r}
        res = get_rx_class_members(params)
        if res['related_ids']:
            rx_concepts.update(res['related_ids'])

print(f"Found {len(rx_concepts)} initial RxNorm concepts")

# Get related concepts
print("Finding related concepts...")
all_rx_list = list(rx_concepts)
count = 0
for rxcui in rx_concepts:
    res = get_related_rxcui(rxcui)
    if res['related_ids']:
        all_rx_list.extend(res['related_ids'])
    count += 1
    if count % 100 == 0:
        print(f"Processed {count}/{len(rx_concepts)} concepts...")

all_rx_final = list(set(all_rx_list))
print(f"Found {len(all_rx_final)} total RxNorm concepts after expansion")

# 5. Compare with existing VO annotations
print("Step 3: Identifying missing RxNorm terms...")
rxnav_data = [get_rxnorm_data(rxcui) for rxcui in all_rx_final]
rxnav_df = pd.DataFrame(rxnav_data)
print(len(rxnav_df), "rows")
# change df column "names" to "label"
rxnav_df.rename(columns={"name": "label"}, inplace=True)


Found 550 initial RxNorm concepts
Finding related concepts...
Processed 100/550 concepts...
Processed 200/550 concepts...
Processed 300/550 concepts...
Processed 400/550 concepts...
Processed 500/550 concepts...
Found 1884 total RxNorm concepts after expansion
Step 3: Identifying missing RxNorm terms...
1884 rows


### 2.1 Encoding and Faiss Index:

In [43]:
rxnav_df.head()

,rxcui,label,status,tty
0,2647547,influenza A virus A/Christchurch/16/2010 (H1N1...,Active,SCDFP
1,2694020,influenza A virus A/turkey/Turkey/1/2005 (H5N1...,Active,SCD
2,798422,hepatitis B surface antigen vaccine 0.02 MG/ML...,Active,SBDC
3,2605731,Flucelvax Quadrivalent 2022-2023,Active,BN
4,2709125,influenza A virus A/Croatia/10136RV/2023 (H3N2...,Active,SCD


In [16]:
# rxnav_df.rename(columns={"name": "label"}, inplace=True)
rxnav_df.head()
# save dataframe as pickle file
rxnav_df.to_pickle("rxnav_data.pkl")

In [13]:
model = load_model(MODEL_ID_LIST[1])

Loading model Alibaba-NLP/gte-Qwen2-1.5B-instruct...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model Alibaba-NLP/gte-Qwen2-1.5B-instruct loaded to CUDA cuda:0 successfully.


In [47]:
vo_df = pd.DataFrame
vo = get_ontology("vo_source/VO_merged2022_10_05.owl").load()
rows = []
for cls in vo.classes():
    name = cls.name
    if name.startswith("VO_"):
        label = cls.label[0] if cls.label else ""
        rows.append({"vo_id": name, "label": label})
vo_df = pd.DataFrame(rows)
vo_df = vo_df.fillna("")
vo_df.to_pickle("vo_data.pkl")

In [14]:
# unpickle the dataframe
vo_df = pd.read_pickle("vo_data.pkl")
vo_df.head() 

,vo_id,label
0,VO_0000001,vaccine
1,VO_0000787,gene - Obsolete
2,VO_0000599,recombinant vaccine vector
3,VO_0000574,route of administration
4,VO_0000333,Protozoa


In [15]:
#VO
vo_texts = vo_df["label"].tolist()
vo_vecs = encode_texts(model, vo_texts, batch_size=64, normalize=True)

Batches:   0%|          | 0/98 [00:00<?, ?it/s]

In [16]:
vo_faiss_ids = vo_df.index.to_numpy(dtype="int64")  # Use DataFrame index as IDs
vo_faiss_index = build_faiss_index(vo_vecs, ids=vo_faiss_ids, use_gpu=True)

Index built with 6231 IDs.


In [ ]:
#RxNorm: Todo: Delete
texts = rxnav_df["label"].tolist()
vecs = encode_texts(model, texts, batch_size=64, normalize=True)
faiss_ids = rxnav_df.index.to_numpy(dtype="int64")  # Use DataFrame index as IDs
faiss_index = build_faiss_index(vecs,ids=faiss_ids, use_gpu=True)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Index built with 1992 IDs.


In [22]:
faiss_cpu = faiss.index_gpu_to_cpu(faiss_index)
faiss.write_index(faiss_cpu, "rxnorm_faiss.bin")

In [ ]:
# del model, vo_vecs
# # clear cache
# torch.cuda.empty_cache()
# import gc
# gc.collect()

0

In [17]:
vo_faiss_cpu = faiss.index_gpu_to_cpu(vo_faiss_index)
faiss.write_index(vo_faiss_cpu, "vo_faiss.bin")

In [88]:
def doc_actions(df):
    for idx, row in df.iterrows():
        yield {
            "_op_type": "index",
            "_index": INDEX,
            "_id"   : f"{row.rxcui}-{idx}",  # unique per synonym
            "rxcui" : row.rxcui,
            "tty"   : row.tty,
            "text"  : row['name']
        }

In [21]:
def vo_doc_actions(df, index_name="vo_bm25"):
    """
    df columns expected: ['vo_id', 'text'].
    ToDo: One row per VO label or synonym. 
    """
    df = df.fillna("")           

    for idx, row in df.iterrows():
        yield {
            "_op_type": "index",
            "_index"  : index_name,
            "_id"     : f"{row.vo_id}-{idx}",    # unique per synonym
            "vo_id"   : row.vo_id,
            # "iri"     : row.iri,
            "text"    : row.label
        }


In [ ]:
# RxNorm
from elasticsearch.helpers import streaming_bulk

for ok, action in streaming_bulk(
        es, doc_actions(rxnav_df), chunk_size=1000, max_retries=5, initial_backoff=2
):
    if not ok:
        print("Failed to index document:", action)


In [22]:
# VO
from elasticsearch.helpers import streaming_bulk
for ok, action in streaming_bulk(
        es, vo_doc_actions(vo_df), chunk_size=1000, max_retries=5, initial_backoff=2
):
    if not ok:
        print("Failed to index document:", action)

In [ ]:
#RxNorm
def dense_candidates(faiss_index, q_vec, meta_df, k=20):
    D, I = faiss_index.search(q_vec.reshape(1,-1), k)
    rows = meta_df.loc[I[0], ["rxcui","label", "status", "tty", ]]
    rows = rows.assign(score=D[0])
    return rows.reset_index(drop=True).to_dict(orient="records")

In [24]:
#VO
def vo_dense_candidates(faiss_index, q_vec, meta_df, k=20):
    D, I = faiss_index.search(q_vec.reshape(1,-1), k)
    rows = meta_df.loc[I[0], ["vo_id","label"]]
    rows = rows.assign(score=D[0])
    return rows.reset_index(drop=True).to_dict(orient="records")

In [ ]:
#RxNorm
def bm25_candidates(es, vo_label: str, k: int = 20, index="rxnorm_bm25"):
    """
    Return top-k lexical hits for a VO label from the RxNorm BM25 index.
    """
    resp = es.search(
        index=index,              # ← keyword
        size=k,                   # ← keyword
        query={                   # ← keyword
            "match": {
                "text": {         # field defined in mappings
                    "query": vo_label,
                    "operator": "or"
                }
            }
        }
    )
    return [
        {
            "rxcui": hit["_source"]["rxcui"],
            "label": hit["_source"]["text"],
            "score": hit["_score"]
        }
        for hit in resp["hits"]["hits"]
    ]

In [25]:
#VO
def vo_bm25_candidates(es, rx_label: str, k: int = 20, index="vo_bm25"):
    """
    Return top-k lexical hits for a VO label from the RxNorm BM25 index.
    """
    resp = es.search(
        index=index,              # ← keyword
        size=k,                   # ← keyword
        query={                   # ← keyword
            "match": {
                "text": {         # field defined in mappings
                    "query": rx_label,
                    "operator": "or"
                }
            }
        }
    )
    return [
        {
            "vo_id": hit["_source"]["vo_id"],
            "label": hit["_source"]["text"],
            "score": hit["_score"]
        }
        for hit in resp["hits"]["hits"]
    ]

In [16]:
VO_terms = ['Flucelvax Quadrivalent 2022-2023 vaccine', 
            '0.5 ML Bordetella pertussis filamentous hemagglutinin vaccine, inactivated 0.04 MG/ML / Bordetella pertussis fimbriae 2/3 vaccine, inactivated 0.01 MG/ML / Bordetella pertussis pertactin vaccine, inactivated 0.006 MG/ML / Bordetella pertussis toxoid vaccine, inactivated 0.04 MG/ML / diphtheria toxoid vaccine, inactivated 30 UNT/ML / Haemophilus influenzae b (Ross strain) capsular polysaccharide meningococcal protein conjugate vaccine 0.006 MG/ML / hepatitis B surface antigen vaccine 0.02 MG/ML / poliovirus vaccine inactivated, type 1 (Mahoney) 58 UNT/ML / poliovirus vaccine inactivated, type 2 (MEF-1) 14 UNT/ML / poliovirus vaccine inactivated, type 3 (Saukett) 52 UNT/ML / tetanus toxoid vaccine, inactivated 10 UNT/ML Injection']

In [96]:
from itertools import chain
from collections import defaultdict

def fuse_hits_rrf(dense_hits, bm25_hits, k=20, rank_bias=60):

    # 1. sort each list by its native score (descending)
    dense_hits  = sorted(dense_hits,  key=lambda x: -x["score"])
    bm25_hits   = sorted(bm25_hits,   key=lambda x: -x["score"])

    # 2. compute RRF contribution from each list
    fused = defaultdict(lambda: {"rrf": 0.0, "origin": set()})

    for rank, hit in enumerate(dense_hits, start=1):
        key = hit["rxcui"]
        fused[key]["rrf"] += 1 / (rank_bias + rank)
        fused[key]["origin"].add("dense")
        # keep a representative label
        fused[key].setdefault("label", hit["label"])

    for rank, hit in enumerate(bm25_hits,  start=1):
        key = hit["rxcui"]
        fused[key]["rrf"] += 1 / (rank_bias + rank)
        fused[key]["origin"].add("bm25")
        fused[key].setdefault("label", hit["label"])

    # 3. turn dict -> list, sort by fused score
    fused_list = [
        {
            "rxcui": key,
            "label": val["label"],
            "fused": val["rrf"],
            "from" : ",".join(sorted(val["origin"]))   # e.g. "bm25,dense"
        }
        for key, val in fused.items()
    ]
    fused_list.sort(key=lambda x: -x["fused"])

    # 4. return top-k
    return fused_list[:k]


In [107]:
k = 5
VO_term = VO_terms[0]  # Example VO term to search
q_vec  = encode_texts(model, [VO_term], batch_size=64, normalize=True)
dense_hits = dense_candidates(faiss_index, q_vec, rxnav_df, k)
lexical_hits = bm25_candidates(es, VO_term, k)
print(f"Lexical hits for '{VO_term}':")
candidates = fuse_hits_rrf(dense_hits, lexical_hits, k=k)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Lexical hits for 'Flucelvax Quadrivalent 2022-2023 vaccine':


In [11]:
# load the faiss index from file
faiss_cpu = faiss.read_index("rxnorm_faiss.bin")

In [13]:
# load index to GPU
res = faiss.StandardGpuResources()
faiss_index = faiss.index_cpu_to_gpu(res, 0, faiss_cpu)

In [18]:
VO_term = VO_terms[0]
q_vec  = encode_texts(model, [VO_term], batch_size=64, normalize=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### 3. Parsing fused results to LLM

In [138]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import json

In [139]:
MODEL_ID = "Qwen/Qwen2-7B-Instruct"
inf_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, use_fast=True, trust_remote_code=True)
inf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.float16, 
    device_map="auto", 
    trust_remote_code=True
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [140]:
def build_prompt(vo, cands):
    # --- System role ---
    sys_msg = ("""\
You are an ontology-alignment assistant. Your task is to map a Vaccine
Ontology (VO) term to the best matching RxNorm clinical drug concept.

* Always choose the **single best** match if it exists.
* If none of the candidates is an acceptable mapping, output "NONE".
* Return answer as **valid JSON object** matching this schema:
  {"rxcui": "<RxCUI | NONE>", "match_type": "exact|narrow|broad|none",
   "justification": "<max 40 words>"}"""
    )

    # --- User role ---
    header = (f"""\
### VO term
ID: {vo["id"]}
Label: {vo["label"]}
Parents: {", ".join(vo["parents"])}
Synonyms: {", ".join(vo["synonyms"])}"""
    )

    # candidate table
    cand_lines = [
        f"{i}. {c['rxcui']} | {c['label']} | fused={c['fused']:.4f} | {c['from']}"
        for i, c in enumerate(cands, 1)
    ]

    # instruction
    user_msg = (header + "### Candidates\n" + "\n".join(cand_lines) +"""\
### Instructions
Choose the best mapping **by RxCUI**. Use the vaccine season, formulation,
and number of viral strains to decide exactness.

Respond with JSON only — no code fences, no extra keys."""
    )

    # combine all parts
    return [
        {"role": "system", "content": sys_msg},
        {"role": "user",   "content": user_msg},
    ]


In [ ]:
vo_term = {
    "id": "VO_0020335",
    "label": "Flucelvax Quadrivalent 2022-2023 vaccine  ",
    "parents": ["human influenza virus vaccine 2022-2023"],
    "synonyms": [""]
}

In [142]:
chat = build_prompt(vo_term, candidates)

prompt_ids = inf_tokenizer.apply_chat_template(
    chat,
    add_generation_prompt=True,   # inserts the assistant “turn header”
    return_tensors="pt"
).to(inf_model.device)   

In [ ]:
out = inf_model.generate(
    prompt_ids,
    max_new_tokens=120,
    temperature=0.1,
    repetition_penalty=1.1,
    do_sample=False
)

assistant_text = inf_tokenizer.decode(out[0][prompt_ids.shape[-1]:],
                                  skip_special_tokens=True).strip()
mapping = json.loads(assistant_text)
assistant_text



The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'{"rxcui": "2605733", "match_type": "exact", "justification": "Best match based on vaccine season, formulation, and number of viral strains."}'

In [1]:
mapping = json.loads(assistant_text)
mapping

NameError: name 'json' is not defined

### 4. Combined Pipeline and Evaluation: 

In [ ]:
# Test different embedding models (MedGemma and other leaderboard models from HF)

### 5. Agentic Workflow

In [2]:
from langchain.agents import initialize_agent
from langchain.tools import Tool

# tools = [
#     Tool(
#         name="search",
#         func=search,
#         description="Search for information"
#     ),
#     Tool(
#         name="lookup",
#         func=lookup,
#         description="Look up information"
#     )
# ]

# agent = initialize_agent(tools)